In [143]:
spark.sql("drop table silver_enriched.delivery_partner_dim")

DataFrame[]

In [140]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_enriched.batch_job_tracker (
        job_name STRING,
        last_processed_version BIGINT,
        run_timestamp TIMESTAMP,
        rows_processed BIGINT
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/enriched/batch_job_tracker'
""")

DataFrame[]

# Silver Enriched: Delivery Partner Dimension (SCD Type 2)

**Job Name:** `delivery_partner_dim_scd2`  
**Source:** `bronze.orders_raw` (via CDF)  
**Target:** `silver_enriched.delivery_partner_dim`  
**SCD2 Trigger:** Change in `partner_name` or `partner_type`

## Import Utils

In [161]:
import sys
sys.path.append("/opt/spark-notebooks/silver_enriched")

from utils import (
    get_last_processed_version,
    save_last_processed_version,
    read_bronze_cdf,
    get_current_bronze_version
)

print("Utils imported ✅")

Utils imported ✅


## Create Table

In [162]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_enriched")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_enriched.delivery_partner_dim (
        delivery_partner_sk BIGINT,
        partner_id STRING,
        partner_name STRING,
        partner_type STRING,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/enriched/delivery_partner_dim'
""")

print("Table ready ✅")

Table ready ✅


## Check Last Processed Version

In [165]:
import importlib
import utils
importlib.reload(utils)

from utils import (
    get_last_processed_version,
    save_last_processed_version,
    read_bronze_cdf,
    get_current_bronze_version
)


In [166]:
JOB_NAME = "delivery_partner_dim_scd2"

last_version = get_last_processed_version(spark, JOB_NAME)
print(f"Last processed version: {last_version}")

Last processed version: 2


## Read Only New Data from Bronze (CDF)

In [ ]:
bronze_df = read_bronze_cdf(spark, last_version)
new_row_count = bronze_df.count()
print(f"New rows to process: {new_row_count}")

No new data. Bronze is still at version 2.
New rows to process: 0


In [169]:
current_version = get_current_bronze_version(spark)
print(current_version)

2


## Parse & Deduplicate Delivery Partner Attributes

In [168]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

if new_row_count > 0:
    partner_schema = StructType([
        StructField("order", StructType([
            StructField("order_time", StringType()),
            StructField("delivery", StructType([
                StructField("delivery_partner", StructType([
                    StructField("partner_id", StringType()),
                    StructField("partner_name", StringType()),
                    StructField("partner_type", StringType())
                ]))
            ]))
        ]))
    ])

    parsed = bronze_df.withColumn("data", from_json(col("raw_payload"), partner_schema))

    incoming_partners = parsed.select(
        col("data.order.delivery.delivery_partner.partner_id").alias("partner_id"),
        col("data.order.delivery.delivery_partner.partner_name").alias("partner_name"),
        col("data.order.delivery.delivery_partner.partner_type").alias("partner_type"),
        col("data.order.order_time").cast("timestamp").alias("order_time")
    )

    # Deduplicate: Keep the LATEST version per partner_id
    w = Window.partitionBy("partner_id").orderBy(col("order_time").desc())

    latest_partners = (
        incoming_partners
        .withColumn("rn", row_number().over(w))
        .filter("rn = 1")
        .drop("rn", "order_time")
    )

    latest_partners.createOrReplaceTempView("incoming_partners")
    print(f"Unique partners to process: {latest_partners.count()}")
    latest_partners.show(truncate=False)
else:
    print("No new data. Skipping. ✅")

No new data. Skipping. ✅


## SCD2 MERGE

In [153]:
if new_row_count > 0:
    existing_count = spark.sql("SELECT COUNT(*) FROM silver_enriched.delivery_partner_dim").collect()[0][0]

    if existing_count == 0:
        print("Seeding dimension...")
        spark.sql("""
            INSERT INTO silver_enriched.delivery_partner_dim
            SELECT
                monotonically_increasing_id() AS delivery_partner_sk,
                partner_id, partner_name, partner_type,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_partners
        """)
        print("Dimension seeded ✅")

    else:
        print("Running SCD2 merge...")
        spark.sql("""
            MERGE INTO silver_enriched.delivery_partner_dim AS target
            USING incoming_partners AS source
            ON target.partner_id = source.partner_id AND target.is_current = true
            WHEN MATCHED AND (
                target.partner_name != source.partner_name OR
                target.partner_type != source.partner_type
            )
            THEN UPDATE SET
                target.is_current   = false,
                target.effective_to = current_timestamp()
        """)
        print("Step 1 done: Closed old versions.")

        spark.sql("""
            INSERT INTO silver_enriched.delivery_partner_dim
            SELECT
                (SELECT COALESCE(MAX(delivery_partner_sk), 0) FROM silver_enriched.delivery_partner_dim)
                    + monotonically_increasing_id() + 1 AS delivery_partner_sk,
                src.partner_id, src.partner_name, src.partner_type,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_partners src
            LEFT ANTI JOIN silver_enriched.delivery_partner_dim dim
                ON src.partner_id = dim.partner_id AND dim.is_current = true
        """)
        print("Step 2 done: Inserted new versions ✅")

    current_version = get_current_bronze_version(spark)
    print(current_version)
    save_last_processed_version(spark, JOB_NAME, current_version, new_row_count)

Seeding dimension...
Dimension seeded ✅
2
Saved bookmark → job: delivery_partner_dim_scd2, version: 2, rows: 5


In [152]:
spark.sql("DESCRIBE HISTORY bronze.orders_raw").toPandas()

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2,2026-03-27 16:37:47.240,None,None,SET TBLPROPERTIES,"{'properties': '{""delta.enableChangeDataFeed"":...",None,None,None,1.0,Serializable,True,{},None,Apache-Spark/3.5.1 Delta-Lake/3.2.0
1,1,2026-03-27 16:27:13.692,None,None,STREAMING UPDATE,"{'epochId': '0', 'outputMode': 'Append', 'quer...",None,None,None,0.0,Serializable,True,"{'numOutputRows': '5', 'numRemovedFiles': '0',...",None,Apache-Spark/3.5.1 Delta-Lake/3.2.0
2,0,2026-03-27 16:26:51.437,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,True,{},None,Apache-Spark/3.5.1 Delta-Lake/3.2.0


In [154]:
spark.sql("select * from silver_enriched.batch_job_tracker").show()

+--------------------+----------------------+--------------------+--------------+
|            job_name|last_processed_version|       run_timestamp|rows_processed|
+--------------------+----------------------+--------------------+--------------+
|delivery_partner_...|                     2|2026-03-27 16:52:...|             5|
+--------------------+----------------------+--------------------+--------------+



In [ ]:
spark.sql("drop table silver_enriched.batch_job_tracker")

## Verify

In [155]:
spark.sql("""
    SELECT delivery_partner_sk, partner_id, partner_name, partner_type,
           is_current, effective_from, effective_to
    FROM silver_enriched.delivery_partner_dim
    ORDER BY partner_id, effective_from
""").show(truncate=False)

+-------------------+----------+------------+------------+----------+--------------------------+-------------------+
|delivery_partner_sk|partner_id|partner_name|partner_type|is_current|effective_from            |effective_to       |
+-------------------+----------+------------+------------+----------+--------------------------+-------------------+
|0                  |DP-01     |Delhivery   |THIRD_PARTY |true      |2026-03-27 16:52:01.506963|9999-12-31 00:00:00|
|1                  |DP-02     |FedEx       |THIRD_PARTY |true      |2026-03-27 16:52:01.506963|9999-12-31 00:00:00|
|2                  |DP-03     |BlueDart    |THIRD_PARTY |true      |2026-03-27 16:52:01.506963|9999-12-31 00:00:00|
+-------------------+----------+------------+------------+----------+--------------------------+-------------------+

